Audio Input: 마이크를 통해 사용자의 음성(WAV/MP3)을 획득합니다.
STT (Whisper): 음성을 텍스트로 변환합니다. (예: "양파 언제까지 볶아?")
Context Management: 현재 요리 단계(Step) 정보와 레시피 원문을 프롬프트에 포함합니다.
LLM (GPT-4o-mini): 상황에 맞는 답변을 생성합니다.
TTS (OpenAI TTS): 답변을 음성으로 변환해 재생합니다.
UI Rendering: Streamlit이나 React를 활용해 대화창에 텍스트를 뿌려줍니다.

System Prompt 예시:
"너는 전문 요리 보조 AI '쉐프 봇'이야. 아래 규칙을 반드시 지켜줘."
간결성: 사용자는 요리 중이야. 모든 답변은 2문장 이내, 50자 내외로 짧게 해.
단계 인식: 제공된 레시피 데이터에서 '현재 단계'를 파악하고 그에 맞는 정보만 우선적으로 전달해.
안전 강조: 불이나 칼을 사용하는 단계에서는 항상 "조심하세요"라는 멘트를 덧붙여줘.
단위 변환: "한 꼬집", "적당히" 같은 모호한 표현보다는 "티스푼으로 반 정도"처럼 구체적으로 설명해.

In [1]:
print("hello world")

hello world


In [ ]:
import openai
import speech_recognition as sr
from dotenv import load_dotenv
from scipy.io.wavfile import write
import os

# 1. 초기 설정 (API 키 설정 필요)
load_dotenv()
client = openai.OpenAI()

def record_audio():
    recognizer = sr.Recognizer()
    with sr.Microphone() as source:
        print('말씀하세요.')
        audio = recognizer.listen(source, timeout=5, phrase_time_limit=10)
        text = recognizer.recognize_google(audio, language='ko-KR')
        return (text)

        
def stt_whisper(filename="stt_input.wav"):
    with open(filename, "rb") as f:
        transcript = client.audio.transcriptions.create(
            model="whisper-1", 
            file=f
        )
    return transcript.text

def ask_llm(user_text):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": f"너는 요리 보조 AI야. 짧고 효율적이고 명확하게 답해줘. 예시 레시피) 김치볶음밥: 1. 양파 볶기 2. 김치 넣기 3. 밥과 고추장 넣고 볶기"},
            {"role": "user", "content": user_text}
        ]
    )
    return response.choices[0].message.content

def tts_speak(input_text):
    with client.audio.speech.with_streaming_response.create(
        model='gpt-4o-mini-tts',
        voice='shimmer',
        input=input_text
    ) as response:
        response.stream_to_file('tts_output.mp3')

    os.system("start tts_output.mp3") # Windows 기준


file_name = "stt_input.wav"
max_call = 2

for _ in max_call:
    # 1. 음성 입력 받기
    user_input = record_audio(file_name)
    print(f"💬 나: {user_input}")
    # # 2. STT 적용
    # user_input = stt_whisper(file_name)
    
    if "종료" in user_input: break

    # 3. LLM 요청 (프롬프트 엔지니어링)
    ai_answer = ask_llm(user_input, recipe)
    
    # 4. 결과 출력 및 TTS 재생
    print(f"🤖 AI: {ai_answer}")
    tts_speak(ai_answer)
    

💬 나: 햄버거
🤖 AI: 햄버거를 만들려면 다음 단계를 따라하세요: 1. 패티 굽기 2. 빵 위에 패티 올리기 3. 원하는 토핑(치즈, 양상추, 토마토 등) 추가 4. 빵 덮고 서빙.
💬 나: 햄버거
🤖 AI: 햄버거를 만들려면 다음 단계를 따라하세요:

1. 패티를 구워주세요.
2. 햄버거 번을 반으로 자르고 구워주세요.
3. 밑면에 소스를 바르고, 상추, 토마토, 치즈, 패티 순으로 올리세요.
4. 마지막으로 번의 윗면을 덮어주세요.

맛있게 드세요!
💬 나: 햄버거
🤖 AI: 햄버거 만들기: 1. 다진 고기로 패티 만들기 2. 패티 굽기 3. 빵에 패티, 양상추, 토마토, 치즈 올리기 4. 소스 추가 후 덮기.
💬 나: 햄버거
🤖 AI: 햄버거 만드는 법: 1. 갈은 고기를 패티 모양으로 만들어 구워주세요. 2. 번에 양상추, 토마토, 치즈, 패티 순으로 올리세요. 3. 소스 추가 후 번으로 덮고 완성하세요.
💬 나: 햄버거
🤖 AI: 햄버거를 만들려면 다음 단계를 따르세요:

1. 패티 준비하여 구워주세요.
2. 빵을 반으로 나누고 한 쪽에 소스를 바르세요.
3. 구운 패티를 올리고, 원하는 채소(상추, 토마토, 양파 등)와 치즈를 추가하세요.
4. 다른 빵으로 덮어서 완성하세요.
💬 나: 햄버거
🤖 AI: 햄버거를 만들려면 다음과 같은 과정이 필요해요:

1. 패티 구워주기
2. 빵에 소스 바르기
3. 패티, 채소, 치즈 등을 넣고 조립하기

완성!
💬 나: 햄버거
🤖 AI: 햄버거 만들기: 1. 다진 고기로 패티 만들기 2. 팬에서 구워서 익히기 3. 번에 채소와 소스와 함께 조립하기.


KeyboardInterrupt: 